# BDB27 — burst vs. jump: how much of a get-off is legs, and how much is reading the snap?

**This notebook is the whole interface.** Set `MODE` in §1 (and `DATA_DIR` only if the data is
not in the usual place), then **Run All**. No terminal, no command to type.

**The design in one line** (`TASK.md`, 2026-11-05). Between-player differences in a defensive
lineman's get-off are split into a **physical** part measured at the Combine (where there is no
snap to react to) and a residual **"jump"** part; the residual is then checked for reliability
across games and agreement with a direct snap-to-onset latency. The outcome is the **tracked
line-of-scrimmage crossing** (`cross_time_s`), never `player_get_off` (validation only).

**Provenance — read this before any number.**

* `MODE = "full"` — the final run. Every stage artefact is stamped `FULL`; only then is a number
  a result.
* `MODE = "sample"` — a plumbing run on the provided, truncated 2023 tracking sample. Every
  artefact is stamped `PLACEHOLDER_SAMPLE: UNVALIDATED SAMPLE OUTPUT`. The headline prints the
  numbers **under a loud `NOT A RESULT` banner** and repeats the qualifier on every group, so a
  sample diagnostic cannot be read as a finding.
* A missing artefact is printed as **`NOT RUN — <field> absent`**. Nothing is guessed, proxied or
  zero-filled.

The stage table in §3 reports every stage's **exit code and elapsed time**. §5 prints the
Phase 5 validation checks, including a synthetic world in which each check *must* fire — so a
check that cannot fail is visible rather than implied.

## 1. Configuration — edit only this cell

* `MODE="sample"` runs the chain on the data this host has, and stamps every artefact
  `UNVALIDATED SAMPLE OUTPUT`.
* `MODE="full"` is the final analysis. It needs the three full tracking files
  (`game_tracking_2023.csv`, `game_tracking_2024.csv`, `game_tracking_2025.csv`) in the data
  directory. If they are absent, the stages that need them are reported **`PENDING FULL RUN`**
  with the missing filenames named; the run continues and never falls back to the sample.
* `DATA_DIR=None` uses the repo default `data/raw`. Set it only when the files live elsewhere
  (a relative path is resolved against the repo root).
* `FORCE_STAGES` rebuilds only the named stages; `[]` reuses checkpoints.
* `CLEAN=True` is destructive but **inert unless `CLEAN_CONFIRM="DELETE"`**. It deletes this
  chain's stage outputs and checkpoints only — never `data/raw` and never the archived outputs.

There is nothing to edit below this cell.

In [ ]:
# ===================== CONFIG — edit only this cell =====================
MODE            = "sample"   # "sample" | "full"
DATA_DIR        = None       # None = repo default data/raw
FORCE_STAGES    = []         # e.g. ["03_game_snaps"]; [] = reuse checkpoints
CLEAN           = False      # destructive; requires CLEAN_CONFIRM == "DELETE"
CLEAN_CONFIRM   = ""         # safety interlock
# ==========================================================================

assert MODE in ("sample", "full"), MODE
assert DATA_DIR is None or isinstance(DATA_DIR, str), DATA_DIR
assert isinstance(FORCE_STAGES, (list, tuple)), FORCE_STAGES
assert all(isinstance(s, str) and s for s in FORCE_STAGES), FORCE_STAGES
assert isinstance(CLEAN, bool), CLEAN
assert isinstance(CLEAN_CONFIRM, str), CLEAN_CONFIRM

print(f"mode            : {MODE}")
print(f"data dir        : {DATA_DIR or 'repo default (data/raw)'}")
print(f"force stages    : {list(FORCE_STAGES) or 'none — reuse checkpoints'}")
print("clean           : "
      + ("ARMED — outputs and checkpoints for this chain will be deleted"
         if CLEAN and CLEAN_CONFIRM == "DELETE"
         else ("IGNORED — set CLEAN_CONFIRM='DELETE' to confirm" if CLEAN
               else "off — existing outputs are preserved")))

## 2. Bootstrap — locate the repo, resolve the data directory, print what is present

This cell walks upward until it sees `config.yaml` and `src/notebook_report.py`, resolves
`DATA_DIR` (writing a scratch run-config with that absolute path when an override is given), and
prints the environment: Python, repo commit, mode, data directory, and a present/absent row for
every input file. A missing file is always named; nothing is silently substituted.

In [ ]:
import importlib
import importlib.util
import shutil
import sys
import time
import traceback
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path


def find_repo_root() -> Path:
    """Walk upward from the notebook's working directory."""
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "notebook_report.py").exists():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/notebook_report.py). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import common as C          # noqa: E402
import notebook_report as R  # noqa: E402

CONFIG_PATH = REPO / "config.yaml"
CFG = C.load_config(CONFIG_PATH)

if DATA_DIR:
    override = Path(DATA_DIR).expanduser()
    DATA_ROOT = (override if override.is_absolute() else (REPO / override)).resolve()
    import copy
    import yaml
    cfg_override = copy.deepcopy(CFG)
    cfg_override.pop("_config_path", None)
    cfg_override["bdb27"]["paths"]["raw"] = str(DATA_ROOT)
    RUN_CFG = REPO / "bdb27_run" / "notebook_config.yaml"
    RUN_CFG.parent.mkdir(parents=True, exist_ok=True)
    RUN_CFG.write_text(yaml.safe_dump(cfg_override, sort_keys=False), encoding="utf-8")
    CONFIG_ARG = ["--config", str(RUN_CFG)]
else:
    DATA_ROOT = C.abs_path(C.block(CFG)["paths"]["raw"])
    RUN_CFG = CONFIG_PATH
    CONFIG_ARG = []

CFG_USED = C.load_config(RUN_CFG)
OUTPUTS = C.paths(CFG_USED, MODE)["outputs"] if MODE in ("sample", "full") else REPO / "outputs"


def _git_sha() -> str:
    try:
        import subprocess
        return subprocess.check_output(
            ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
            text=True, stderr=subprocess.DEVNULL).strip()
    except Exception as exc:                       # git absent, not a repo, no permission
        return f"unavailable ({type(exc).__name__})"


def _files(key: str):
    try:
        return C.source_path(CFG_USED, key)
    except KeyError:
        return []


FILE_KEYS = ["combine_tracking", "combine_results", "players", "games", "player_play",
             "game_tracking_sample", "game_tracking_full"]
FULL_TRACKING = _files("game_tracking_full")
FULL_TRACKING_MISSING = [p.name for p in FULL_TRACKING if not p.exists()]

print("ENVIRONMENT")
print("-" * 88)
print(f"  python      : {sys.version.split()[0]}")
print(f"  repo        : {REPO}")
print(f"  commit      : {_git_sha()}")
print(f"  mode        : {MODE}")
print(f"  data dir    : {DATA_ROOT}")
print(f"  run config  : {RUN_CFG}")
print("")
print("INPUT FILES")
print("-" * 88)
for key in FILE_KEYS:
    files = _files(key)
    if not files:
        print(f"  {key:<22} : NOT CONFIGURED")
        continue
    for path in files:
        mark = "present" if path.exists() else "ABSENT"
        size = ""
        if path.exists():
            size = f"  ({path.stat().st_size / 1024 ** 2:.1f} MB)"
        print(f"  {key:<22} : {mark:<7} {path.name}{size}")
print("-" * 88)
if FULL_TRACKING_MISSING:
    print("  full tracking missing: " + ", ".join(FULL_TRACKING_MISSING))

## 3. The stage chain

`TASK.md` fixes the order: `01_audit → 02_combine_physical_time → 03_game_snaps →
03b_reliability → 04_models → 05_validate`. Every stage below is implemented and runs here;
§5 prints what the validation stage found.

| # | stage | reads | writes |
|---|---|---|---|
| 1 | `01_audit` | `combine_*`, `players`, `games`, `player_play`, tracking probe | `outputs/audit/` — items a–i, `gate.json` |
| 2 | `02_combine_physical_time` | `combine_tracking`, `combine_results`, `players`, `player_play` | `outputs/phase2/` — per-player physical time vs distance, attempt QC, calibration |
| 3 | `03_game_snaps` | `player_play`, game tracking | `outputs/phase3/` — one row per DL pass-rush snap (`snaps.parquet`) |
| 3b | `03b_reliability` | `phase3/snaps.parquet`, complete label-side files | `outputs/phase3b/` — per-player reliability, selection rules |
| 4 | `04_models` | `phase2/physical_time_by_player.csv`, `phase3/snaps.parquet` | `outputs/phase4/` — the variance decomposition |
| 5 | `05_validate` | `phase4/decomposition.json`, `phase2/`, `phase3/snaps.parquet` | `outputs/validation/` — permutation, placebo, mechanical, snap-quality and selection checks, plus the synthetic can-fail worlds |

Every stage is invoked through **its own `main()`**, and the stage module plus its `src/` helpers
are reloaded from disk before each call, so a `git pull` cannot leave stale code in the kernel.

**Full-mode degradation.** If `game_tracking_{2023,2024,2025}.csv` are absent, `01_audit` and
`03_game_snaps` are reported `PENDING FULL RUN` with the missing filenames; `03b_reliability`,
`04_models` and `05_validate` are `PENDING FULL RUN` because their upstream table cannot exist.
A stale sample parquet is never used as a substitute. `02_combine_
physical_time` is attempted and its gate decision reported honestly. The loop never aborts and
never substitutes the sample.

In [ ]:
# The stages this notebook runs, in TASK.md order.
STAGE_ORDER = ["01_audit", "02_combine_physical_time", "03_game_snaps",
               "03b_reliability", "04_models", "05_validate"]

#: Stages that cannot be honest without the three full tracking files.
FULL_TRACKING_STAGES = {"01_audit", "03_game_snaps"}
#: Stages whose inputs cannot exist without the Phase 3 per-snap table.
#: 05_validate reads the Phase 4 fit, which is built from that table.
NEEDS_PHASE3 = {"03b_reliability", "04_models", "05_validate"}

#: Reloaded from disk before every stage call (helpers first).
HELPER_MODULES = ("common", "schema", "physical", "stats", "models", "reliability", "power")


def _purge_bytecode(module_name: str) -> None:
    """Delete the cached .pyc so a reload must recompile from source (D46 trap)."""
    module = sys.modules.get(module_name)
    path = getattr(module, "__file__", None) if module is not None else None
    if not path or not str(path).endswith(".py"):
        return
    try:
        cached = importlib.util.cache_from_source(str(path))
    except (NotImplementedError, ValueError):
        return
    try:
        Path(cached).unlink()
    except OSError:
        pass


def reload_helpers() -> list:
    importlib.invalidate_caches()
    reloaded = []
    for name in HELPER_MODULES:
        if name in sys.modules:
            _purge_bytecode(name)
            importlib.reload(sys.modules[name])
            reloaded.append(name)
    return reloaded


def load_stage(module_name: str):
    """Import a stage fresh from the file on disk (never a cached module object)."""
    reload_helpers()
    path = SRC / f"{module_name}.py"
    if not path.exists():
        raise FileNotFoundError(f"stage module absent: {path}")
    spec = importlib.util.spec_from_file_location(f"bdb27_nb_{module_name}", path)
    if spec is None or spec.loader is None:
        raise ImportError(f"cannot load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def run_stage(module_name: str, extra_args: list) -> dict:
    """Run one stage via its own main(); return rc, seconds, captured output."""
    started = time.perf_counter()
    argv = ["--mode", MODE, *CONFIG_ARG, *extra_args]
    buffer = StringIO()
    rc, error = 0, ""
    old_argv = sys.argv
    try:
        module = load_stage(module_name)
        sys.argv = [f"{module_name}.py", *argv]
        with redirect_stdout(buffer), redirect_stderr(buffer):
            result = module.main(argv)
        rc = 0 if result is None else int(result)
    except SystemExit as exc:
        rc = 0 if exc.code in (None, 0) else int(exc.code)
    except C.GateRefused as exc:
        rc, error = 4, f"REFUSED: {exc}"
        print(error, file=buffer)
    except BaseException as exc:                    # noqa: BLE001 - report, never hide
        rc, error = 1, f"{type(exc).__name__}: {exc}"
        traceback.print_exc(file=buffer)
    finally:
        sys.argv = old_argv
    return {"stage": module_name, "rc": rc, "seconds": time.perf_counter() - started,
            "error": error, "output": buffer.getvalue()}


# ---- CLEAN: delete this chain's outputs + checkpoints, only when armed ----
if CLEAN and CLEAN_CONFIRM != "DELETE":
    print("CLEAN=True is IGNORED: set CLEAN_CONFIRM='DELETE' to confirm deletion.")

if CLEAN and CLEAN_CONFIRM == "DELETE":
    targets = [OUTPUTS / "audit", OUTPUTS / "phase2", OUTPUTS / "phase3",
               OUTPUTS / "phase3b", OUTPUTS / "phase4", OUTPUTS / "figures"]
    checkpoint_dir = C.paths(CFG_USED, MODE)["checkpoints"]
    print("CLEAN ARMED — deleting this chain's outputs and checkpoints")
    for target in targets:
        if target.exists():
            count = sum(1 for p in target.rglob("*") if p.is_file())
            print(f"  {target}  ({count} files)")
            shutil.rmtree(target)
    for path in (checkpoint_dir.glob(f"*__{MODE}__*.json")):
        print(f"  {path}")
        path.unlink()
    run_log = C.abs_path(C.block(CFG_USED)["paths"]["run_log"])
    if run_log.exists():
        print(f"  {run_log}")
        run_log.unlink()

# ---- run the chain ---------------------------------------------------------
results = []
for stage in STAGE_ORDER:
    is_pending, reason = False, ""
    if MODE == "full" and FULL_TRACKING_MISSING:
        if stage in FULL_TRACKING_STAGES:
            is_pending = True
            reason = "needs " + ", ".join(FULL_TRACKING_MISSING)
        elif stage in NEEDS_PHASE3:
            # A full-mode 03_game_snaps cannot run without the files, so its
            # downstream tables cannot be rebuilt on full data. A stale sample
            # parquet is never used as a substitute.
            is_pending = True
            reason = ("needs 03_game_snaps, which needs "
                      + ", ".join(FULL_TRACKING_MISSING))
    if is_pending:
        results.append({"stage": stage, "status": "PENDING FULL RUN", "rc": None,
                        "seconds": 0.0, "error": reason, "output": "", "note": ""})
        continue
    outcome = run_stage(stage, ["--force"] if stage in FORCE_STAGES else [])
    if outcome["rc"] == 0:
        outcome["status"] = "OK"
    elif outcome["rc"] == 4:
        outcome["status"] = "REFUSED (gate)"
    elif outcome["rc"] == 3:
        outcome["status"] = "NOT BUILT"
    else:
        outcome["status"] = "FAILED"
    output_lower = outcome["output"].lower()
    outcome["note"] = ("checkpoint reused — use FORCE_STAGES to rebuild"
                       if "checkpoint" in output_lower else "")
    results.append(outcome)

print("STAGE RESULTS")
print("=" * 88)
for row in results:
    elapsed = f"{row['seconds']:6.1f} s" if row["rc"] is not None else "     —  "
    rc_text = f"exit {row['rc']}" if row["rc"] is not None else "   —  "
    note = row.get("note") or ""
    print(f"  {row['stage']:<26} {rc_text:<8} {elapsed:<10} {row['status']}"
          + (f"  ({note})" if note else ""))
    if row["error"]:
        for line in str(row["error"]).splitlines():
            print(f"      {line}")
print("=" * 88)

failures = [r for r in results if r["status"] == "FAILED"]
if failures:
    print("FAILED STAGES — captured output (last 25 lines each)")
    for row in failures:
        print("-" * 88)
        print(row["stage"])
        for line in row["output"].strip().splitlines()[-25:]:
            print("  " + line)
else:
    print("No stage failed. (PENDING FULL RUN is not a failure.)")

## 4. Headline — the block to send back

Everything here is read from `outputs/phase4/decomposition.json`; no number is typed into this
notebook. It prints, in this order:

1. **the provenance banner**, taken from the artefact itself — `PLACEHOLDER_SAMPLE` gets a loud
   `UNVALIDATED SAMPLE OUTPUT — NOT A RESULT` banner, `FULL` gets `PROVENANCE: FULL`;
2. the **model and estimator**, including the measurement-error treatment;
3. **pooled, edge and interior separately**: the between-player split (phys vs jump `u`), the
   total-variance split (phys, jump, within-player remainder), each with its 95% interval; `b`
   with its interval; the jump SD in ms; reliability rows; latency-agreement rows; the rung used;
   the boundary flags (a zero share is printed as zero and never replaced by the naive rung);
4. the **resolution limit** from the `resolution` block (smallest detectable jump SD ≈ 52–54 ms)
   stated as a resolution limit, **not** a pass/fail threshold;
5. the physical-interpolation coverage counter, and the same counter as a one-line caveat
   **immediately after the between-player phys share**, because that share is estimated only on the
   reps that carry a Combine curve;
6. an **integrity block** before the model block: which upstream stages failed or refused in this
   run, the calibration record's status, provenance and blocker, and — when calibration is not
   `ESTIMATED` from a `FULL` record — the plain consequence that the physical times carry **no
   calibration offset** and the numbers are produced without one. A `FULL` stamp on the
   decomposition records the data mode, not that every upstream stage completed, and the block
   says so. This block **reports and does not gate**: nothing is blocked, and no gate is changed.

A missing file or field prints `NOT RUN — <field> absent`. In `MODE="sample"` the numbers are
printed as **sample diagnostics** under the banner, with the qualifier repeated on every group,
so no sample value can be read as a finding.

In [ ]:
print(R.render_headline(OUTPUTS, MODE, results))

## 5. Validation — can each check fail?

Everything here is read from `outputs/validation/validation.json`; no number is typed into
this notebook. It prints, in this order:

1. **permutation** — the observed physical share, the permutation distribution, the one-sided
   Monte-Carlo p and the number of resamples, and the artefact's own reading. If the observed
   share sits at the zero boundary the block says the check is **uninformative on this data**
   rather than implying it passed;
2. **placebo** — the real physical share beside the height/weight pseudo-phys share;
3. **mechanical** — base, depth removed, alignment removed, both removed, the onset-threshold
   sweep, and the stance line stating that stance subsets are **not possible** (no stance label
   in any released file, B8);
4. **snap quality** — the share dropped at both the tracked and fitted levels, and the shares
   on the clean subset;
5. **selection** — the snap-count spread and the unweighted vs equal-snap shares;
6. **the resolution statement**;
7. **the synthetic "can this check fail" table** — one row per detector with its
   `detector_fired` value. This is the evidence that the checks are not decoration, so it is
   printed in full, not summarised away.

A missing file or field prints `NOT RUN — <field> absent`. In `MODE="sample"` the whole block
sits under the loud `NOT A RESULT` banner: a sample diagnostic is never validation of a real
result.

In [ ]:
print(R.render_validation(OUTPUTS, MODE))

## 6. Figures (max 4 — `TASK.md` line 115)

The task allows **four** figures. Each is rendered **only from `FULL`-provenance outputs**; in
sample mode, or when an input is missing, the cell prints why that figure was skipped. No figure
is ever built from placeholder data.

1. **Variance split with intervals, edge vs interior** — the primary result made visible.
2. **Combine-predicted physical time vs observed get-off at mean depth**, with the players
   furthest off the diagonal named.
3. **Reliability and latency agreement of jump** — split-half, model-based, and the two
   correlations, for pooled / edge / interior.
4. **One rusher's rep-level story** — every rep against that player's own physical curve, with
   his estimated `u`.

In [ ]:
figures = R.render_figures(OUTPUTS, OUTPUTS / "figures", MODE, dpi=150)
for message in figures["messages"]:
    print(message)

if figures["paths"]:
    from IPython.display import Image, display
    for path in figures["paths"]:
        display(Image(filename=path))

## 7. Writeup — rendered from the same artefacts, or not at all

`src/writeup_render.py` fills `notes/writeup-draft.md`'s `{{PENDING_FULL_RUN: ...}}` tokens from
`decomposition.json` and `validation.json`. This cell prints the renderer's **status**, the
**rendered word count**, and every token it could not resolve.

* `REFUSED` — a present artefact is not `FULL`. The cell prints the refusal and **writes nothing**:
  a sample run can never put a number into the document.
* `UNRESOLVED` — some fields are absent. Each is named, and nothing is written (the write is
  strict), so an incomplete document cannot be mistaken for the deliverable.
* `OK` — every token resolved; the rendered markdown is written to
  `outputs/writeup/writeup_rendered.md`.

The count is the rendered body only. The draft's scaffolding after
`<!-- END OF WRITEUP BODY -->` is not rendered and not counted against the 2,000-word cap.


In [ ]:
# The writeup is rendered from the same FULL artefacts the headline reads.
# The renderer refuses sample provenance outright: on a non-FULL stamp it prints
# the refusal and writes nothing. The write is strict, so an UNRESOLVED render
# is reported and not written either.
writeup_render = load_stage("writeup_render")
writeup = writeup_render.render(REPO / "notes" / "writeup-draft.md", OUTPUTS)
print(writeup_render.report(writeup))
written = writeup_render.write(writeup, OUTPUTS / "writeup" / "writeup_rendered.md",
                               strict=True)
print(f"rendered file: {written if written else 'not written'}")


## 8. If something looks wrong, send these

Send the files themselves, not screenshots of a terminal.

1. **§4 headline output** — the plain-English result or the reason it is `NOT RUN`.
2. **§3 stage table** — exit code and elapsed time for every stage; `PENDING FULL RUN` names the
   missing file; `FAILED` means the captured output below the table names the cause.
3. **§5 validation output** — every check, its reading, and the synthetic detector table.
4. `outputs/phase4/decomposition.json` — the artefact the headline reads.
5. `outputs/phase4/player_effects.csv` — per-player `u`, its SE, and the physical deviation.
6. `outputs/phase4/model_form.json` — the exact model, estimator, convergence and constraints.
7. `outputs/validation/validation.json` — every Phase 5 check and its synthetic can-fail world.
8. `outputs/phase3b/reliability_by_player.csv` — per-player reps, games, mean, within SD and
   reliability.
9. `outputs/run_log.csv` — wall time and peak RAM per stage.
10. `notes/blockers.md` — every open data/design gap.
11. If a stage is `FAILED`, the captured output printed under the table in §3.

**What this notebook deliberately does not do.** It does not fall back to the sample in full
mode, does not present a sample number as a result, and does not print a validation check
without the synthetic world that shows the check can fail. The chain it runs does not read a pressure field (D112), does not use `player_get_off`
(D110), and adds no blocker or stance term (D98, B8).